# Project 1

Welcome to the first NLP1 project. The goal of the first tutorial was to get you familiar with text segmentation and simple text classification for topic detection.
Given the skills you have developed in the first week, you will now be applying them within a different dataset and task.

We are recommending a set of tasks and associated datasets; using those, you will be experimenting on problems introduced during the tutorials. You will be applying hands-on the experiments you have already explored in the last tutorial, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and analysing errors. 

The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

As the tutorials will be building upon your knowledge throughout the weeks, the task and dataset you choose now will be the ones you use in projects 2 and 3; and the ones you will be finally writing about in your final blog post.  

## Get to work!

The next section has a checklist to help you not lose track of what needs to be done. Here we have a more informal guide, with prompts and questions for you. 


1. [Choose a dataset](http://nlp-illc.github.io/nlp1/). Spend some time understanding how the dataset looks and some statistics of it (e.g. how many instances of each classification label we have?)
2. Remember what we learnt during the tutorial: to develop any classifier of the kind we saw, we needed a way to segment text. Do you want to use the tokeniser we have developed? Do you need to train a new one? If you do, what hyperparameters shall you use? What data shall you use to train it? Consider that your dataset may be too small to train a tokeniser! Think whether you will need a larger resource of similar nature to train a tokeniser.
3. Start thinking how to represent your documents into features. We have experimented with token occurences and frequencies. Can you think of any other ways, possibly? 
4. We had experimented with Logistic Regression and a shallow neural network. You can experiment with and assess different architectures now and their hyper parameters (e.g. add layers, possibly change the activation functions, drop out rates).
5. Evaluate your models! Was the evaluation method adequate for your new experimental setup? Do you want to assess other aspects of performance?
6. Analyse the various models you assessed and try to decide on which is the best.

Be mindful that you will be using your experiments to write up your final blog, so make sure they are clean and neat!

You can write your own code and you can reuse code from the tutorial and its packages or from other sources. You may use coding assistants if you like, but use it mindfully as you take full responsibility for this kind of (semi-)automated coding. Also, keep a record of the kinds of things you delegate to any form of assistant, as you will be asked to disclose this in your blog post (not because we will pass a value judgement on it, but because it is a widespread and good practice to document such things).

You may use jupyter notebooks or python files, it's up to you as we will not directly assess your code. But, once again, be organised, as we will ask you to share a reproducible version of your experiments along with your blog post (again, because this is how real research is publicised). 

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

## Checklist

What **we expect you to do**:

- [ ] Compare different systems of text classification for your choice of data/task.
- Your comparisons should cover:
    - [ ] the two representation spaces in the tutorial (token indicators and bag-of-words) and an additional one of your choice.
    - [ ] the two kinds of architectures in the tutorial (log-linear and single-hidden-layer MLP) and an additional one of your choice (such as a modification of the MLP).
- [ ] Your comparisons must be based on cross-validation and a relevant metric (accuracy or else).
- [ ] You should make use of Bayesian estimation to help you analyse results in relation to a region of practical equivalence (ROPE), and you can motivate the ROPE size yourself.

What you need to **bring to the oral assessment**:
* You will be asked questions about the points above, you can bring a slide deck with up to five slides (one for each of the following topics _data and metrics_, _features_, _architectures_, _results_, _extra_); you _will not_ give a presentation, rather, these slides are to be thought of as "backup slides" that you can use to guide your answers to the questions the TA will ask you.
* You may also be asked questions about the tutorial, for those, you don't need to prepare any deliverable (a good way to prepare is to work through the tutorial and reflect on the questions there).

What you should **keep in mind** given that you will eventually write a blog post about your work throughout the course:
* Organise your code;
* Summarise your motivations and findings;
* Document your sources and any form of AI-assistance you use.

---
# Our project: Name that LLM

## 0. Reproducibility

Every experiment below is seeded so anyone can re-run it and get the same numbers.

- **During development** use one seed (`MODE = "dev"`), so iterations stay fast.
- **For reported results** switch to `MODE = "report"`: every experiment runs once per seed (5 seeds × k folds). We report the mean and a 95% interval across runs. `BenavoliCorrelatedT(..., runs=len(SEEDS))` in `part1.analysis` accounts for the fold overlap.
- Call `set_seed(seed)` **right before** every split / training call. `part1.smlp.train_neural_network_model` does not seed PyTorch itself, so without this the MLP weights and batch order change on every run.
- The seed controls the fold split (`cross_validation_split(..., seed=seed)`) **and** the model's initialisation, so each run is a different split + initialisation.

In [ ]:
# ---- Reproducibility setup: run this cell first ----
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import torch

warnings.simplefilter(action="ignore", category=FutureWarning)

# Paths (this notebook lives in root/part1)
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
import part1  # noqa: E402  (course helpers: experiment, loglin, smlp, analysis, ...)

DATA_DIR = ROOT / "data" / "name-that-llm"
GRAPHS_DIR = ROOT / "graphs"
GRAPHS_DIR.mkdir(exist_ok=True)
RESULTS_DIR = ROOT / "outputs" / "part1"  # gitignored
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# Seeds
MODE = "dev"  # "dev" -> 1 seed (fast) | "report" -> 5 seeds (for the final numbers)
DEV_SEEDS = [0]
REPORT_SEEDS = [0, 1, 2, 3, 4]
SEEDS = DEV_SEEDS if MODE == "dev" else REPORT_SEEDS
NUM_FOLDS = 5
SPLIT_SEED = 42


def set_seed(seed: int) -> None:
    """Seed every RNG we use (Python, NumPy, PyTorch) and make torch deterministic where possible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(
        seed
    )  # only affects subprocesses; kernel hash seed is fixed at startup
    torch.use_deterministic_algorithms(True, warn_only=True)


def summarise_runs(scores, confidence=0.95):
    """Mean and t-based CI over independent runs (e.g. one mean accuracy per seed)."""
    from scipy import stats

    scores = np.asarray(scores, dtype=float)
    mean = scores.mean()
    if len(scores) < 2:
        return mean, (np.nan, np.nan)
    half = stats.t.ppf((1 + confidence) / 2, len(scores) - 1) * stats.sem(scores)
    return mean, (mean - half, mean + half)


set_seed(SEEDS[0])
print(
    f"MODE={MODE} | SEEDS={SEEDS} | folds={NUM_FOLDS} | torch {torch.__version__} | numpy {np.__version__}"
)

## 1. Data: Name that LLM

**What they ask:** choose a dataset, spend time understanding it and compute some statistics (e.g. how many instances of each label).

**Our task:** given a text, predict which LLM (`model` column) wrote it. Kaggle competition: [name-that-llm](https://www.kaggle.com/competitions/name-that-llm).

**Rubric / oral-assessment points** (slide: _data and metrics_):
- How many classes, how many examples per class? Is it balanced, and what does that imply for the metric (accuracy vs. macro-F1)?
- What is `prompt_id`? The same prompt is answered by several models, so folds should be grouped by prompt to avoid leakage (see Step 5).
- Text length per model: could a classifier "cheat" on length alone?
- Which models are near-duplicates (same family, e.g. `gemini-3.x-flash`, `gpt-5.6-*`)? Expect confusions there.
- `test.csv` only has a handful of rows (the real Kaggle test set is hidden), so all evaluation is via cross-validation on `train.csv`.

In [ ]:
import pandas as pd
from IPython.display import display

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
print(train_df.shape, test_df.shape)
display(train_df.head())

# Label distribution
display(train_df["model"].value_counts().to_frame("count"))

# Prompts: how many, and how many models answer each one?
print("unique prompts:", train_df["prompt_id"].nunique())
display(train_df.groupby("prompt_id")["model"].nunique().describe())

# Length per model (characters / whitespace tokens)
train_df["n_chars"] = train_df["text"].str.len()
train_df["n_words"] = train_df["text"].str.split().str.len()
display(train_df.groupby("model")[["n_chars", "n_words"]].mean().sort_values("n_words"))

# Outcome space: map label names <-> integer ids (as in the tutorial)
outcome_space = part1.clslib.NamedClasses(sorted(train_df["model"].unique()))
pairs = [(t, outcome_space.index(m)) for t, m in zip(train_df["text"], train_df["model"])]
print(len(outcome_space), "classes")

### Data Splitting

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=SPLIT_SEED)
train_idx, test_idx = next(gss.split(train_df, y=train_df["model"], groups=train_df["prompt_id"]))

train_split_df = train_df.iloc[train_idx].reset_index(drop=True)
test_split_df = train_df.iloc[test_idx].reset_index(drop=True)

train_split_df.to_csv(DATA_DIR / "split_train.csv", index=False)
test_split_df.to_csv(DATA_DIR / "split_test.csv", index=False)

# Verify the split is correct
assert train_split_df["prompt_id"].nunique() == 0.8 * train_df["prompt_id"].nunique()
assert test_split_df["prompt_id"].nunique() == 0.2 * train_df["prompt_id"].nunique()
assert set(train_split_df["prompt_id"]).isdisjoint(test_split_df["prompt_id"])

print(len(train_split_df), len(test_split_df))

In [ ]:
OpenAI = "OpenAI"
Llama = "Llama"
Google = "Google"
Anthropic = "Anthropic"
Qwen = "Qwen"
Glm = "GLM"
Kimi = "Kimi"

model_family_map = {
    "qwen3.8-max": Qwen,
    "gemini-3.1-pro-preview": Google,
    "gemini-3.7-flash": Google,
    "gemini-3.6-flash": Google,
    "gemini-3.8-flash": Google,
    "gpt-5.6-terra": OpenAI,
    "gpt-5.6-sol": OpenAI,
    "gpt-5.5": OpenAI,
    "gpt-5.6-luna": OpenAI,
    "gpt-6-astra": OpenAI,
    "claude-opus-5": Anthropic,
    "claude-sonnet-5": Anthropic,
    "claude-opus-4-6": Anthropic,
    "claude-opus-4-8": Anthropic,
    "claude-fable-5": Anthropic,
    "glm-5.3": Glm,
    "glm-5.3-flash": Glm,
    "kimi-k3": Kimi,
}

### EDA

In [ ]:
eda_df = train_split_df.copy()
eda_df["family"] = eda_df["model"].map(model_family_map)
text = eda_df["text"]

eda_df["n_chars"] = text.str.len()
eda_df["n_words"] = text.str.split().str.len()
eda_df["chars_per_word"] = eda_df["n_chars"] / eda_df["n_words"].replace(0, np.nan)

stats = (
    eda_df.groupby(["family", "model"])[["n_words", "n_chars", "chars_per_word"]]
    .agg(["mean", "median", "std"])
    .round(2)
)

display(stats)

family_stats = (
    eda_df.groupby("family")[["n_words", "n_chars", "chars_per_word"]]
    .agg(["mean", "median", "std"])
    .round(2)
)

display(family_stats)

In [ ]:
from eda import add_length_features, analyse_feature, summarise_by_group

EDA_GRAPHS_DIR = GRAPHS_DIR / "eda"
EDA_GRAPHS_DIR.mkdir(exist_ok=True)

In [ ]:
eda_df = add_length_features(train_split_df)
eda_df["family"] = eda_df["model"].map(model_family_map)

display(summarise_by_group(eda_df, ["n_words", "n_chars", "chars_per_word"]))

In [ ]:
res_len = analyse_feature(
    eda_df,
    "n_words",
    "Words per response",
    save_dir=EDA_GRAPHS_DIR,
    prefix="resp_len",
    log_scale_model=True,
)

In [ ]:
res_cpw = analyse_feature(
    eda_df,
    "chars_per_word",
    "Characters per word",
    save_dir=EDA_GRAPHS_DIR,
    prefix="chars_per_word",
)

In [ ]:
from eda import feature_tests, style_feature_tests

tests = feature_tests(eda_df, ["n_words", "chars_per_word", "letters_per_word"])
display(style_feature_tests(tests))

In [ ]:
from part1.eda import (
    PUNCT_MARKS,
    QUOTE_MARKS,
    add_length_features,
    add_punctuation_features,
    feature_tests,
    plot_model_heatmap,
    punct_cols,
    quote_style_table,
    quote_style_test,
    style_feature_tests,
    usage_share,
)

eda_df = add_length_features(train_split_df)
eda_df["family"] = eda_df["model"].map(model_family_map)
eda_df = add_punctuation_features(eda_df)

rates = eda_df.groupby(["family", "model"])[punct_cols()].mean()
plot_model_heatmap(
    rates,
    labels={c: c.removesuffix("_per100") for c in punct_cols()},
    title="Punctuation marks per 100 words (mean per response)",
    save_path=EDA_GRAPHS_DIR / "punct_per100_heatmap.png",
)

In [ ]:
qt = quote_style_table(eda_df)
plot_model_heatmap(
    qt[["curly", "straight", "mixed"]],
    scale="global",
    cmap="Purples",
    title="Quote style (responses containing quotes)",
    cbar_label="share of responses",
    save_path=EDA_GRAPHS_DIR / "quote_style_heatmap.png",
)
display(qt)
print(quote_style_test(eda_df))
display(usage_share(eda_df, [f"n_{q}" for q in QUOTE_MARKS]).round(2))

In [ ]:
punct_tests = feature_tests(eda_df, punct_cols())
display(style_feature_tests(punct_tests))

### Reflections — Step 1

> _Write here: what we did, why (vs. reasonable alternatives), pros/cons, surprises, open questions for the TA._

-

## 2. Tokenisation

**What they ask:** to build any classifier we need a way to segment text. Do we reuse the tutorial's tokeniser, or train a new one? If new, with which hyperparameters (vocab size, min frequency, pre-tokeniser), and on which data? Our dataset may be too small to train a tokeniser, so would a larger resource of similar nature be needed?

**Rubric / oral-assessment points** (slide: _features_; tutorial questions may come up too):
- Why BPE (subwords) vs. whitespace / character tokenisation for _authorship_ of LLM text? Punctuation, formatting and casing may carry the stylistic signal, so should we keep them?
- Vocab size trade-off: larger vocabularies give more specific features but sparser counts on ~2.7k documents.
- Training data for the tokeniser: train on all of `train.csv` (risk: it sees validation folds), only on the training folds, or an external corpus? Justify the choice.
- Unknown tokens / coverage: what fraction of tokens become `[UNK]`?

Helpers: `part1.experiment.train_tokenizer(corpus, vocab_size, min_frequency)`, `part1.bpe` (tutorial's BPE).

### Reflections — Step 2

> _Write here: what we did, why (vs. reasonable alternatives), pros/cons, surprises, open questions for the TA._

-

## 3. Document representations (features)

**What they ask:** think about how to turn documents into features. The tutorial used token occurrences (indicators) and frequencies (bag-of-words). Can we think of other ways?

**Rubric (checklist):** compare **the two representation spaces from the tutorial (token indicators and bag-of-words) plus one more of our choice**.

**Oral-assessment points** (slide: _features_):
- What does each representation capture, and what does it throw away (order, length, frequency)?
- Candidates for the third representation: TF-IDF, character n-grams, word n-grams, stylometric features (sentence length, punctuation rates, markdown usage), or combinations. Why this one for _LLM authorship_?
- Dimensionality / sparsity, and how the choice interacts with the classifier.

Helpers: `part1.clslib.TokenIndicator` (indicator encoder); bag-of-words and the third encoder are ours to write as `part1.clslib.Encoder` subclasses.

### Reflections — Step 3

> _Write here: what we did, why (vs. reasonable alternatives), pros/cons, surprises, open questions for the TA._

-

## 4. Architectures

**What they ask:** the tutorial used logistic regression (log-linear) and a shallow neural network. Experiment with different architectures and their hyperparameters (e.g. more layers, different activation functions, dropout rates).

**Rubric (checklist):** compare **log-linear, single-hidden-layer MLP, plus one more architecture of our choice** (e.g. a modification of the MLP).

**Oral-assessment points** (slide: _architectures_):
- What does the MLP add over the log-linear model, and does this dataset size (~2.7k docs, ~18 classes) support it?
- Why our third architecture, and what hypothesis does it test?
- Which hyperparameters did we tune, how (on what data, to avoid tuning on the evaluation folds), and why those values?

Helpers: `part1.loglin.fit`, `part1.smlp.SparseMLP`, `part1.smlp.train_neural_network_model`, `part1.experiment.cross_validation_experiment(..., model_option="logistic" | "mlp")`.

### Reflections — Step 4

> _Write here: what we did, why (vs. reasonable alternatives), pros/cons, surprises, open questions for the TA._

-

## 5. Evaluation

**What they ask:** evaluate the models. Was the evaluation method adequate for our new experimental setup? Do we want to assess other aspects of performance?

**Rubric (checklist):** comparisons must be **based on cross-validation and a relevant metric** (accuracy or other).

**Oral-assessment points** (slide: _data and metrics_ / _results_):
- Is accuracy appropriate here (balanced classes?) or should we also report macro-F1 / per-class F1 / confusion matrix?
- **Leakage:** `cross_validation_split` shuffles texts at random, so answers to the same prompt land in both train and validation. Do we need grouped folds by `prompt_id`? Compare both and explain the difference.
- How many folds and seeds, and why? (Here: `NUM_FOLDS` folds × `SEEDS`.)
- Other aspects: training/inference time, calibration, which models get confused with each other.

In [ ]:
# from collections import OrderedDict

# # One entry per system, e.g. "logistic_indicator" -> list over seeds of cross_validation_experiment results
# results = OrderedDict()


# def run_cv(name, encoder, model_option, **kwargs):
#     """Run k-fold CV once per seed, seeding right before each split + training run."""
#     results[name] = []
#     for seed in SEEDS:
#         set_seed(seed)
#         splits = part1.experiment.cross_validation_split(pairs, k=NUM_FOLDS, seed=seed)
#         set_seed(seed)
#         res = part1.experiment.cross_validation_experiment(
#             splits,
#             outcome_space,
#             encoder,
#             model_option,
#             save_to=str(RESULTS_DIR / name / f"seed-{seed}"),
#             **kwargs,
#         )
#         results[name].append(res)
#     per_seed = [np.mean(list(r["accuracies"].values())) for r in results[name]]
#     mean, ci = summarise_runs(per_seed)
#     print(f"{name}: acc {mean:.4f}  95% CI {ci}")


# # TODO: e.g.
# # run_cv("logistic_indicator", indicator_enc, "logistic", loglin_args=LOGLIN_ARGS)
# # run_cv("mlp_indicator", indicator_enc, "mlp", smlp_args=SMLP_ARGS)

### Reflections — Step 5

> _Write here: what we did, why (vs. reasonable alternatives), pros/cons, surprises, open questions for the TA._

-

## 6. Analysis: which system is best?

**What they ask:** analyse the models we assessed and decide which is best.

**Rubric (checklist):** use **Bayesian estimation to analyse results relative to a region of practical equivalence (ROPE)**, and **motivate the ROPE size** ourselves.

**Oral-assessment points** (slide: _results_, plus _extra_):
- What do P(A better), P(equivalent), P(B better) mean, and how do they depend on the ROPE?
- Why our ROPE size (e.g. what accuracy difference would matter in practice for this task)?
- Is the best system clearly better, or practically equivalent to a simpler one? If equivalent, which would we pick and why?
- Error analysis: which LLMs get confused (confusion matrix), and do the errors make sense (same family / provider)?
- Extra slide ideas: grouped-vs-random folds, ensembles, feature inspection (most predictive tokens per model).

Helpers: `part1.analysis.make_data_frame`, `plot_box_and_points`, `BenavoliCorrelatedT(sys_a, sys_b, runs=len(SEEDS)).plot(rope)` / `.probs(rope)`.

### Reflections — Step 6

> _Write here: what we did, why (vs. reasonable alternatives), pros/cons, surprises, open questions for the TA._

-

## AI-assistance & sources log

We must disclose this in the blog post. Keep a running list: what was delegated to an assistant, and which external sources/code were used.

| Date | Who | Tool / source | What for |
|---|---|---|---|
| 2026-11-02 | Cecce | Claude Code | Notebook skeleton (this structure + reproducibility cell) |